# Imports

In [ ]:
import numpy as np
import pandas as pd

from pipeline.data_exploration import plot_targets, plot_acf_pacf, run_hypothesis_tests, plot_boxplpot, \
    plot_heatmap_correlation
from utils.constants import CAMS_VARS, ERA5_VARS, STATES, TARGETS, WEATHER_AGG_STATS, CONSUMERS_SLUG
from utils.dataset_manipulators import merge_datasets


# Data Visualization

In [ ]:
dfs = {}
dfs_final = {}

for consumption in CONSUMERS_SLUG:
    df = pd.read_csv(f"../data/processed/dataset_{consumption}.csv")
    df_final = pd.read_csv(f"../data/processed/dataset_final_{consumption}.csv")

    df["Date"] = pd.to_datetime(df["Date"], format="%m/%Y")
    df_final["Date"] = pd.to_datetime(df_final["Date"], format="%m/%Y")

    df = df.set_index("Date")
    df_final = df_final.set_index("Date")

    dfs[consumption] = df
    dfs_final[consumption] = df_final

df_merged = merge_datasets(dfs)
df_merged_clean = merge_datasets(dfs_final)



## Paraná

In [ ]:
df_pr = df_merged[df_merged["State_PR"] == 1]
plot_targets(df_pr, "_PR")

## Santa Catarina

In [ ]:
df_sc = df_merged[df_merged["State_SC"] == 1]
plot_targets(df_sc, "_SC")

## Rio Grande do Sul

In [ ]:
df_rs = df_merged[df_merged["State_RS"] == 1]
plot_targets(df_rs, "_RS")

# Statistical Summary

In [ ]:
resumo_list = []

for state in STATES:
    df_state = df_merged[df_merged[f"State_{state}"] == 1]
    
    for target in TARGETS:
        serie = df_state[target]
        resumo_list.append({
            "State": state,
            "Consumption": target,
            "Frequency": "Monthly",
            "Data Points": int(serie.count()),
            "Mean": np.round(serie.mean(), 2),
            "Standard Deviation": np.round(serie.std(), 2),
            "Min": serie.min(),
            "Max": serie.max(),
        })

df_resumo = pd.DataFrame(resumo_list)
df_resumo = df_resumo.set_index(["State", "Consumption"])
df_resumo.to_csv("../paper/materials/statistical_summary.csv")
df_resumo

## Paraná

In [ ]:
plot_acf_pacf(df_pr, state_name="PR", lags=12)
df_tests_pr = run_hypothesis_tests(df_pr, state_name="PR")
display(df_tests_pr)


## Santa Catarina

In [ ]:
plot_acf_pacf(df_sc, state_name="SC", lags=12)
df_tests_sc = run_hypothesis_tests(df_sc, state_name="SC")
display(df_tests_sc)


## Rio Grande do Sul

In [ ]:
plot_acf_pacf(df_rs, state_name="RS", lags=12)
df_tests_rs = run_hypothesis_tests(df_rs, state_name="RS")
display(df_tests_rs)


## Compiled Hypothesis Tests

In [ ]:
df_tests_all = pd.concat([df_tests_pr, df_tests_sc, df_tests_rs])
df_tests_all.to_csv("../paper/materials/hypothesis_tests_all.csv")
df_tests_all

# Boxplots

## Without Data Clean


In [ ]:
plot_boxplpot(df_merged, "boxplot_original_data")

## With Data Clean

In [ ]:
plot_boxplpot(df_merged_clean, "boxplot_cleaned_data")

# Correlation Analysis

## Consumption vs Consumers

In [ ]:
for target in TARGETS:
    df = dfs_final[target.lower()]
    target_df = target.lower()
    var_cols = sorted([col for col in df.columns if col.startswith(f"{target} Consumers")])

    plot_heatmap_correlation(
        df[[target]+var_cols],
        title=f"Correlation Matrix - {target} Consumption vs Consumers"
    )

## Consumption vs Lags

In [ ]:
for target in TARGETS:
    df = dfs_final[target.lower()]
    target_df = target.lower()
    var_cols = sorted([col for col in df.columns if col.startswith(f"{target}_LAG")])

    plot_heatmap_correlation(
        df[[target] + var_cols],
        title=f"Correlation Matrix - {target} Consumption vs Consumers"
    )


## Consumption vs Climate Data

In [ ]:
for target in TARGETS:
    target_df = target.lower()

    df = dfs_final[target.lower()]
    for var in CAMS_VARS + ERA5_VARS:
        for suffix in WEATHER_AGG_STATS:
            var_cols = sorted([col for col in df.columns if col.startswith(f"{var}_") and col.endswith(f"{suffix}")])
            if not var_cols:
                continue

            plot_heatmap_correlation(
                df[[target] + var_cols],
                title=f"Correlation Matrix - Consumption vs {var} ({suffix})"
            )


# Final Datasets

In [ ]:
dfs_final["residential"]

In [ ]:
dfs_final["commercial"]

In [ ]:
dfs_final["industrial"]

In [ ]:
dfs_final["other"]